In [1]:
#Автоматическая перезагрузка при изменениях
%load_ext autoreload
%autoreload 2

In [2]:
# Для MPS рекомендуется включить fallback на CPU для неподдерживаемых операций
import os

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

In [3]:
from src.data_utils import load_and_clear_data, load_or_tokenize, create_sequences, create_data_split, build_vocab
from src.constants import BATCH_SIZE
from src.next_token_dataset import NextTokenDataset
from src.lstm_model import LSTMLanguageModel
from src.lstm_train import train_model
from src.lstm_test import test_model
from src.generate_examples_lstm import generate_examples

import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

/Users/anna/Documents/ds_projects/sprint_2_rnn/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Подготовка данных

In [4]:
texts = load_and_clear_data()

Загружено 10000 текстов, после очистки осталось 9979 текстов.
Примеры очищенных текстов: ['a thats a bummer. you shoulda got david carr of third day to do it. d', 'is upset that he cant update his facebook by texting it... and might cry as a result school today also. blah!', 'i dived many times for the ball. managed to save 50 the rest go out of bounds', 'my whole body feels itchy and like its on fire', 'no, its not behaving at all. im mad. why am i here? because i cant see you all over there.']


In [5]:
tokenized_texts = load_or_tokenize(texts)

Загрузка токенизированных данных из data/tokenized_texts.pkl...


In [6]:
X, Y = create_sequences(tokenized_texts)

In [7]:
display(X[1:2])
display(Y[1:2])

[['a',
  'thats',
  'a',
  'bummer',
  '.',
  'you',
  'shoulda',
  'got',
  'david',
  'carr',
  'of',
  'third',
  'day',
  'to',
  'do',
  'it',
  '.',
  'd']]

[['thats',
  'a',
  'bummer',
  '.',
  'you',
  'shoulda',
  'got',
  'david',
  'carr',
  'of',
  'third',
  'day',
  'to',
  'do',
  'it',
  '.',
  'd',
  '<EOS>']]

In [8]:
X_train, Y_train, X_val, Y_val, X_test, Y_test = create_data_split(X, Y) 

Train: 127046, Val: 15881, Test: 15881


In [9]:
lengths = [len(tokens) for tokens in tokenized_texts]
print(f"Средняя длина: {sum(lengths) / len(lengths):.1f}")
print(f"Медиана: {sorted(lengths)[len(lengths)//2]}")
print(f"90-й перцентиль: {sorted(lengths)[int(len(lengths)*0.9)]}")

Средняя длина: 16.9
Медиана: 16
90-й перцентиль: 28


In [10]:
# Создание словаря
word2idx, idx2word = build_vocab(tokenized_texts)

vocab_size = len(word2idx)
print(f"Vocabulary size: {vocab_size}")

Vocabulary size: 13784


In [11]:
# Создание датасетов
train_dataset = NextTokenDataset(X_train, Y_train, word2idx)
val_dataset = NextTokenDataset(X_val, Y_val, word2idx)
test_dataset = NextTokenDataset(X_test, Y_test, word2idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

### Тренировка модели LSTM

In [12]:
device = (
        "cuda" if torch.cuda.is_available() else
        "mps" if torch.backends.mps.is_available() else
        "cpu"
    )
print(f"Using device: {device}")

print("torch.backends.mps.is_available:", torch.backends.mps.is_available())
print("torch.backends.mps.is_built:", torch.backends.mps.is_built())

Using device: mps
torch.backends.mps.is_available: True
torch.backends.mps.is_built: True


In [13]:
model = LSTMLanguageModel(vocab_size, word2idx, idx2word).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [15]:
train_losses, val_losses = train_model(model, train_loader, val_loader, criterion, optimizer, idx2word, word2idx)

Epoch 1/10 [Train]: 100%|██████████| 993/993 [02:28<00:00,  6.68it/s, loss=4.8236]


Epoch 1, Train Loss: 5.4222, Val loss: 4.4809
Val ROUGE-1: 0.0781, Val ROUGE-2: 0.0133 


Epoch 2/10 [Train]: 100%|██████████| 993/993 [02:34<00:00,  6.44it/s, loss=3.6689]


Epoch 2, Train Loss: 4.0854, Val loss: 3.4853
Val ROUGE-1: 0.1636, Val ROUGE-2: 0.0666 


Epoch 3/10 [Train]: 100%|██████████| 993/993 [02:44<00:00,  6.03it/s, loss=3.0418]


...

Epoch 8, Train Loss: 2.3206, Val loss: 2.1140
Val ROUGE-1: 0.5686, Val ROUGE-2: 0.4474 


Epoch 9/10 [Train]: 100%|██████████| 993/993 [02:48<00:00,  5.91it/s, loss=2.1775]


Epoch 9, Train Loss: 2.2415, Val loss: 2.0507
Val ROUGE-1: 0.5948, Val ROUGE-2: 0.4667 


Epoch 10/10 [Train]: 100%|██████████| 993/993 [02:49<00:00,  5.86it/s, loss=2.2391]


Epoch 10, Train Loss: 2.1786, Val loss: 2.0004
Val ROUGE-1: 0.5679, Val ROUGE-2: 0.4537 


In [17]:
test_model(model, test_loader, idx2word, word2idx, device, criterion)


ФИНАЛЬНАЯ ОЦЕНКА НА ТЕСТОВОЙ ВЫБОРКЕ (10%)
Test Loss: 1.9903
Test ROUGE-1: 0.6969, Test ROUGE-2: 0.5911 


In [20]:
# Разнообразные фразы
seed_phrases = [
    "i feel",
    "i am feeling so",
    "i love",
    "today is a",
    "i hate when",
    "so sad",
    "i just want"
]

# Генерация примеров
generate_examples(
    model=model, 
    seed_texts=seed_phrases, 
    word2idx=word2idx,
    max_length=10,
    temperature=0.8
)


ПРИМЕРЫ ГЕНЕРАЦИИ МОДЕЛИ

🌱 Seed: "i feel"
🤖 Output: i feel like i sent her
------------------------------------------------------------

🌱 Seed: "i am feeling so"
🤖 Output: i am feeling so happy today . am gon na be okay .
------------------------------------------------------------

🌱 Seed: "i love"
🤖 Output: i love that horrible . what you want ?
------------------------------------------------------------

🌱 Seed: "today is a"
🤖 Output: today is a thumbsup
------------------------------------------------------------

🌱 Seed: "i hate when"
🤖 Output: i hate when we go as the bowl .
------------------------------------------------------------

🌱 Seed: "so sad"
🤖 Output: so sad . i just couldnt meet him on thursday im leaving
------------------------------------------------------------

🌱 Seed: "i just want"
🤖 Output: i just want my new phone
------------------------------------------------------------
